In [10]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import os
import json
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
# 엑셀 파일을 읽어와서 DataFrame으로 변환합니다.
# '파일경로.xlsx' 부분을 실제 엑셀 파일 경로로 변경해주세요.
df = pd.read_excel('filght_data.xls')

# 변환된 DataFrame의 첫 5개 행을 출력하여 확인합니다.
print(df.head())

     운항편명      항공사   출발시간  목적지 터미널  월  화  수  목  금  토  일                   운항기간
0  3U3974     사천항공  14:50  TFU  T1  N  Y  N  Y  N  Y  N  2025.04.01~2025.10.25
1  3U7024     사천항공  16:45  CAN  T1  Y  Y  Y  Y  Y  Y  Y  2025.06.11~2025.10.25
2  3U7032     사천항공  15:15  CAN  T1  Y  Y  Y  Y  Y  Y  Y  2025.03.30~2025.10.25
3  3U7034     사천항공  10:20  CAN  T1  Y  Y  Y  Y  Y  Y  Y  2025.07.01~2025.10.25
4   5J129  세부퍼시픽항공  22:00  CEB  T1  Y  Y  Y  Y  Y  Y  Y  2025.07.13~2025.08.08


In [12]:
# 2. 데이터 전처리
# '운항기간'을 시작일과 종료일로 분리하고 날짜 형식으로 변환합니다.
df[['운항시작일', '운항종료일']] = df['운항기간'].str.split('~', expand=True)
df['운항시작일'] = pd.to_datetime(df['운항시작일'], format='%Y.%m.%d')
df['운항종료일'] = pd.to_datetime(df['운항종료일'], format='%Y.%m.%d')

In [13]:
# 3. 필터링 로직
# 조회할 기간 설정
start_date = pd.to_datetime('2025-07-18')
end_date = pd.to_datetime('2025-07-25')
target_dates = pd.date_range(start_date, end_date)

# 요일 한글 이름과 데이터프레임의 요일 컬럼명을 매핑
day_of_week_map = {
    0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'
}

# 결과를 저장할 리스트 초기화
filtered_flights = []

# 설정된 기간의 각 날짜에 대해 반복
for target_date in target_dates:
    # 해당 날짜의 요일을 구함 (월요일=0, 일요일=6)
    day_name = day_of_week_map[target_date.weekday()]

    # 각 항공편에 대해 조건 확인
    for index, row in df.iterrows():
        # 조건 1: 조회 날짜가 운항 기간에 포함되는지 확인
        is_in_period = (row['운항시작일'] <= target_date <= row['운항종료일'])
        # 조건 2: 해당 요일에 운항하는지 확인
        is_operating_day = (row[day_name] == 'Y')

        if is_in_period and is_operating_day:
            flight_info = {
                '운항편명': row['운항편명'],
                '항공사': row['항공사'],
                '출발시간': row['출발시간'],
                '목적지': row['목적지'],
                '터미널': row['터미널'],
                '운행 날짜': target_date.strftime('%Y-%m-%d'),
                '요일': day_name
            }
            filtered_flights.append(flight_info)

In [14]:
# 4. 결과 리스트를 데이터프레임으로 변환
result_df = pd.DataFrame(filtered_flights)

In [15]:
# 5. 최종 결과 출력
print("### 필터링된 항공편 정보")
print(result_df)

### 필터링된 항공편 정보
        운항편명      항공사   출발시간  목적지 터미널       운행 날짜 요일
0     3U7024     사천항공  16:45  CAN  T1  2025-07-18  금
1     3U7032     사천항공  15:15  CAN  T1  2025-07-18  금
2     3U7034     사천항공  10:20  CAN  T1  2025-07-18  금
3      5J129  세부퍼시픽항공  22:00  CEB  T1  2025-07-18  금
4      5J187  세부퍼시픽항공  01:15  MNL  T1  2025-07-18  금
...      ...      ...    ...  ...  ..         ... ..
8733  ZH3308     심천항공  08:30  CAN  T1  2025-07-25  금
8734  ZH3310     심천항공  14:50  PEK  T1  2025-07-25  금
8735   ZH628     심천항공  14:50  WUX  T1  2025-07-25  금
8736   ZH632     심천항공  16:40  SZX  T1  2025-07-25  금
8737   ZH634     심천항공  14:40  SZX  T1  2025-07-25  금

[8738 rows x 7 columns]


In [ ]:
import psycopg2
from psycopg2.extras import execute_values

# 1. PostgreSQL 연결(커넥션 유지)
conn = psycopg2.connect(
    host='host',
    database='postgres',
    user='user',
    password='password',
    port=5432
)

cursor = conn.cursor()

In [17]:
table_name = 'silver.api_flight_schedule'
print(f"'{table_name}' 테이블의 모든 데이터를 삭제합니다...")
cursor.execute(f"DELETE FROM {table_name};")

# 데이터프레임을 튜플의 리스트 형태로 변환
tuples = [tuple(x) for x in result_df.to_numpy()]

# 삽입할 컬럼들을 문자열로 만듦
# hourly_df의 컬럼 이름을 DB 테이블에 맞게 변경 (예시)
result_df.columns = ['flight_number', 'airline', 'scheduled_time', 'destination', 'terminal', 'scheduled_date', 'day_of_week']

# 데이터프레임의 컬럼명으로 cols 문자열을 동적으로 생성
cols = ','.join(list(result_df.columns))
    
# INSERT 쿼리 템플릿 생성
query = f"INSERT INTO {table_name} ({cols}) VALUES %s"

# execute_values를 사용해 데이터 삽입
execute_values(cursor, query, tuples)
    
# 변경 사항을 데이터베이스에 커밋
conn.commit()
    
print(f"'{table_name}' 테이블에 {len(result_df)}개의 행이 성공적으로 삽입되었습니다.")

'silver.api_flight_schedule' 테이블의 모든 데이터를 삭제합니다...
'silver.api_flight_schedule' 테이블에 8738개의 행이 성공적으로 삽입되었습니다.


In [18]:
cursor.close()
conn.close()